In [ ]:
# =========================
# Colab Script (Drive path version)
# - Train/Test are pre-split CSVs
# - K_EDGE: 1..max_edges (step=1)
#   * O  : original features only (fixed)
#   * F  : edge features only (top-k by ranking)
#   * OF : original + top-k edge features (original always included)
#
# Metrics: F1, AUROC, AUPRC, Brier, ECE
# Outputs: results_O.csv / results_F.csv / results_OF.csv
# columns: SET, DAG, MODEL, K_EDGE, AUROC, AUPRC, F1, Brier, ECE
# =========================

!pip -q install lightgbm xgboost

import os
import random
import numpy as np
import pandas as pd

from typing import Dict, List, Tuple

from google.colab import drive
drive.mount('/content/drive')

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# =========================
# Config
# =========================
RANDOM_STATE = 42
VAL_SIZE = 0.2
ECE_BINS = 15

# FFMLP config
RUN_FFMLP = True       # 느리면 False
BATCH_SIZE = 2048
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5


# =========================
# Paths (Google Drive)
# =========================
BASE_DIR = "/content/drive/MyDrive/bank_failure_prediction"

# base (no edge features)
DATA_BASE = {
    "train": os.path.join(BASE_DIR, "data_with_features_train.csv"),
    "test":  os.path.join(BASE_DIR, "data_with_features_test.csv"),
}

# per-alg (has edge features)
DATA_DAG = {
    "NOTEARS": {"train": os.path.join(BASE_DIR, "data_with_features_NOTEARS_train.csv"),
                "test":  os.path.join(BASE_DIR, "data_with_features_NOTEARS_test.csv")},
    "PC":      {"train": os.path.join(BASE_DIR, "data_with_features_PC_train.csv"),
                "test":  os.path.join(BASE_DIR, "data_with_features_PC_test.csv")},
    "GES":     {"train": os.path.join(BASE_DIR, "data_with_features_GES_train.csv"),
                "test":  os.path.join(BASE_DIR, "data_with_features_GES_test.csv")},
    "GOLEM":   {"train": os.path.join(BASE_DIR, "data_with_features_GOLEM_train.csv"),
                "test":  os.path.join(BASE_DIR, "data_with_features_GOLEM_test.csv")},
}

OUT_DIR = os.path.join(BASE_DIR, "results_tables")
os.makedirs(OUT_DIR, exist_ok=True)


# =========================
# Seed
# =========================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)


# =========================
# Metrics
# =========================
def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    y_true = y_true.astype(int)
    y_prob = np.clip(y_prob, 0.0, 1.0)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        if not np.any(mask):
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)

def brier_score(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    y_true = y_true.astype(float)
    y_prob = np.clip(y_prob, 0.0, 1.0)
    return float(np.mean((y_prob - y_true) ** 2))

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AUROC": float(roc_auc_score(y_true, y_prob)),
        "AUPRC": float(average_precision_score(y_true, y_prob)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Brier": brier_score(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob, n_bins=ECE_BINS),
    }


# =========================
# Data utils
# =========================
def detect_target_col(df: pd.DataFrame) -> str:
    candidates = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found. Tried {candidates}")

def split_train_val(X_train_all: np.ndarray, y_train_all: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_all, y_train_all,
        test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=y_train_all
    )
    return X_train, X_val, y_train, y_val

def to_numpy(df: pd.DataFrame, cols: List[str]) -> np.ndarray:
    return df[cols].to_numpy(dtype=np.float32)

def is_index_col(c: str) -> bool:
    return c.startswith("Unnamed") or c.lower() in {"index", "_index"}

def get_original_cols(df_any: pd.DataFrame, target_col: str) -> List[str]:
    return [c for c in df_any.columns
            if c != target_col and (not c.startswith("edge_")) and (not is_index_col(c))]


# =========================
# Edge ranking (NO GEXF)
# - edge weight(|w|) 접근 불가 → train에서 edge feature의 std 큰 순서로 랭킹
# =========================
def get_edge_cols(df_alg: pd.DataFrame, alg: str) -> List[str]:
    prefix = f"edge_{alg}__"
    cols = [c for c in df_alg.columns if c.startswith(prefix)]
    if len(cols) == 0:
        cols = [c for c in df_alg.columns if c.startswith("edge_")]
    return cols

def rank_edges_by_train_std(df_alg_train: pd.DataFrame, edge_cols: List[str]) -> List[str]:
    if len(edge_cols) == 0:
        return []
    stds = df_alg_train[edge_cols].astype(float).std(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return stds.sort_values(ascending=False).index.tolist()

def pick_k_edge_cols(edge_ranked_cols: List[str], k_edge: int) -> List[str]:
    return edge_ranked_cols[:min(int(k_edge), len(edge_ranked_cols))]


# =========================
# Models
# =========================
def fit_predict_logit(X_train, y_train, X_val, y_val, X_test):
    clf = LogisticRegression(max_iter=2000, solver="lbfgs")
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

def fit_predict_rf(X_train, y_train, X_val, y_val, X_test):
    clf = RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

def fit_predict_xgb(X_train, y_train, X_val, y_val, X_test):
    clf = xgb.XGBClassifier(
        tree_method="hist",
        device="cuda",  # GPU 가능하면 사용, CPU면 내부적으로 fallback 가능

        n_estimators=800,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr

def fit_predict_lgbm(X_train, y_train, X_val, y_val, X_test):
    clf = lgb.LGBMClassifier(
        device="gpu",   # GPU 가능하면 사용
        gpu_platform_id=0,
        gpu_device_id=0,

        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )
    clf.fit(X_train, y_train)
    val_prob = clf.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, val_prob)
    test_prob = clf.predict_proba(X_test)[:, 1]
    return test_prob, thr


# =========================
# FFMLP (PyTorch)
# =========================
class FFMLP(nn.Module):
    def __init__(self, in_dim: int, hidden: List[int] = [128, 64], dropout: float = 0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

@torch.no_grad()
def predict_proba_mlp(model: nn.Module, X: np.ndarray, device: str) -> np.ndarray:
    model.eval()
    dl = DataLoader(TensorDataset(torch.tensor(X)), batch_size=4096, shuffle=False)
    probs = []
    for (xb,) in dl:
        xb = xb.to(device)
        logits = model(xb)
        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)
    return np.concatenate(probs, axis=0)

def train_mlp(X_train, y_train, X_val, y_val) -> Tuple[FFMLP, float]:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = FFMLP(in_dim=X_train.shape[1]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_ds = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=4096, shuffle=False)

    best_val = float("inf")
    best_state = None
    bad = 0

    for _ in range(EPOCHS):
        model.train()
        for xb, yb in train_dl:
            xb = xb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        model.eval()
        vlosses = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                vlosses.append(loss_fn(logits, yb).item())
        v = float(np.mean(vlosses))

        if v < best_val - 1e-6:
            best_val = v
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    y_val_prob = predict_proba_mlp(model, X_val, device=device)
    thr = best_f1_threshold(y_val, y_val_prob)
    return model, thr


# =========================
# Evaluation runner
# =========================
def eval_all_models(X_train_all, y_train_all, X_test, y_test, run_ffmlp: bool = True) -> List[Dict[str, float]]:
    X_train, X_val, y_train, y_val = split_train_val(X_train_all, y_train_all)
    out = []

    # LightGBM
    prob, thr = fit_predict_lgbm(X_train, y_train, X_val, y_val, X_test)
    out.append({"MODEL": "LightGBM", **compute_metrics(y_test, prob, thr)})

    # XGBoost
    prob, thr = fit_predict_xgb(X_train, y_train, X_val, y_val, X_test)
    out.append({"MODEL": "XGBoost", **compute_metrics(y_test, prob, thr)})

    # FFMLP
    if run_ffmlp:
        model_mlp, thr = train_mlp(
            X_train.astype(np.float32), y_train.astype(np.float32),
            X_val.astype(np.float32), y_val.astype(np.float32)
        )
        device = "cuda" if torch.cuda.is_available() else "cpu"
        prob = predict_proba_mlp(model_mlp, X_test.astype(np.float32), device=device)
        out.append({"MODEL": "FFMLP", **compute_metrics(y_test, prob, thr)})

    # Logit
    prob, thr = fit_predict_logit(X_train, y_train, X_val, y_val, X_test)
    out.append({"MODEL": "Logit", **compute_metrics(y_test, prob, thr)})

    # RF
    prob, thr = fit_predict_rf(X_train, y_train, X_val, y_val, X_test)
    out.append({"MODEL": "RF", **compute_metrics(y_test, prob, thr)})

    return out


# =========================
# Load base once
# =========================
for k in ["train", "test"]:
    if not os.path.exists(DATA_BASE[k]):
        raise FileNotFoundError(f"Missing base {k}: {DATA_BASE[k]}")

df_base_train = pd.read_csv(DATA_BASE["train"], low_memory=False)
df_base_test  = pd.read_csv(DATA_BASE["test"],  low_memory=False)

target_col = detect_target_col(df_base_train)
orig_cols = get_original_cols(df_base_train, target_col)

y_train_base = df_base_train[target_col].to_numpy(dtype=np.int64)
y_test_base  = df_base_test[target_col].to_numpy(dtype=np.int64)

missing_in_test = [c for c in orig_cols + [target_col] if c not in df_base_test.columns]
if missing_in_test:
    raise ValueError(f"[BASE] Missing columns in test: {missing_in_test}")

X_train_base = to_numpy(df_base_train, orig_cols)
X_test_base  = to_numpy(df_base_test,  orig_cols)


# =========================
# Run (O / F / OF)
# =========================
rows_O, rows_F, rows_OF = [], [], []

for alg, paths in DATA_DAG.items():
    print(f"\n[ALG] {alg}")

    for k in ["train", "test"]:
        if not os.path.exists(paths[k]):
            raise FileNotFoundError(f"Missing {alg} {k}: {paths[k]}")

    df_alg_train = pd.read_csv(paths["train"], low_memory=False)
    df_alg_test  = pd.read_csv(paths["test"],  low_memory=False)

    if target_col not in df_alg_train.columns or target_col not in df_alg_test.columns:
        raise ValueError(f"[{alg}] target_col '{target_col}' not found in alg train/test")

    y_train_alg = df_alg_train[target_col].to_numpy(dtype=np.int64)
    y_test_alg  = df_alg_test[target_col].to_numpy(dtype=np.int64)

    # ---------- O ----------
    o_metrics = eval_all_models(X_train_base, y_train_base, X_test_base, y_test_base, run_ffmlp=RUN_FFMLP)
    for m in o_metrics:
        rows_O.append({"SET":"O","DAG":alg,"K_EDGE":0, **m})

    # ---------- Edge ranking ----------
    edge_cols_all = get_edge_cols(df_alg_train, alg)
    edge_ranked_cols = rank_edges_by_train_std(df_alg_train, edge_cols_all)
    max_edges = len(edge_ranked_cols)
    print(f"[INFO] edge cols: {max_edges}")

    if max_edges == 0:
        # OF with 0 edges (same as O but keep format)
        of_metrics = eval_all_models(X_train_base, y_train_base, X_test_base, y_test_base, run_ffmlp=RUN_FFMLP)
        for m in of_metrics:
            rows_OF.append({"SET":"OF","DAG":alg,"K_EDGE":0, **m})
        continue

    # Ensure orig cols exist in alg dfs (for OF)
    missing_orig_in_alg_test = [c for c in orig_cols + [target_col] if c not in df_alg_test.columns]
    if missing_orig_in_alg_test:
        raise ValueError(f"[{alg}] Missing original columns in alg test: {missing_orig_in_alg_test}")

    # ---------- K_EDGE: 1..max ----------
    for k_edge in range(1, max_edges + 1):
        edge_cols_used = pick_k_edge_cols(edge_ranked_cols, k_edge)

        # F
        X_train_F = to_numpy(df_alg_train, edge_cols_used)
        X_test_F  = to_numpy(df_alg_test,  edge_cols_used)

        f_metrics = eval_all_models(X_train_F, y_train_alg, X_test_F, y_test_alg, run_ffmlp=RUN_FFMLP)
        for m in f_metrics:
            rows_F.append({"SET":"F","DAG":alg,"K_EDGE":k_edge, **m})

        # OF
        of_cols = orig_cols + edge_cols_used
        X_train_OF = to_numpy(df_alg_train, of_cols)
        X_test_OF  = to_numpy(df_alg_test,  of_cols)

        of_metrics = eval_all_models(X_train_OF, y_train_alg, X_test_OF, y_test_alg, run_ffmlp=RUN_FFMLP)
        for m in of_metrics:
            rows_OF.append({"SET":"OF","DAG":alg,"K_EDGE":k_edge, **m})


# =========================
# Save CSVs
# =========================
def finalize_table(rows: List[dict], set_name: str) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    df = df[["SET","DAG","MODEL","K_EDGE","AUROC","AUPRC","F1","Brier","ECE"]]
    df = df.sort_values(["DAG","K_EDGE","AUROC","AUPRC","MODEL"], ascending=[True, True, False, False, True]).reset_index(drop=True)
    df["SET"] = set_name
    return df

tbl_O  = finalize_table(rows_O, "O")
tbl_F  = finalize_table(rows_F, "F")
tbl_OF = finalize_table(rows_OF, "OF")

out_o  = os.path.join(OUT_DIR, "results_O.csv")
out_f  = os.path.join(OUT_DIR, "results_F.csv")
out_of = os.path.join(OUT_DIR, "results_OF.csv")

tbl_O.to_csv(out_o, index=False)
tbl_F.to_csv(out_f, index=False)
tbl_OF.to_csv(out_of, index=False)

print("\n[DONE] saved:")
print(out_o)
print(out_f)
print(out_of)

display(tbl_O.head(10))
display(tbl_F.head(10))
display(tbl_OF.head(10))
